In [21]:
import pandas as pd
from pathlib import Path   
import numpy as np

In [22]:
result_path = Path("../results/downstream_task")
bias_types = ["less_positive_class"]
metrics = ["AUROC", "AUPRC"]
method_name_replacer = {"uniform": "Uniform", 
                        "kmm": "KMM",
                        "psa": "PSA",
                        "mrs-forest": "MRS",
                        "fw-mrs-temperature": "FW-MRS",
                        "fw-mrs-temperature-svm": "FW-MRS$_{SVM}$",
                        "soft-mrs-linear": "Soft-MRS",
                        }
data_set_replacer = {"diabetes": "Diabetes",
    "folktables_employment": "Employment", 
                     "folktables_income": "Income",
                     "bank_marketing": "Bank Marketing",
                     "hr_analytics": "HR Analytic",
                     "german_credit": "German Credit", 
                     "breast_cancer": "Breast Cancer", 
                     "loan_prediction": "Loan",
                    }

In [ ]:
aurocs = []
auprcs = []
dict_list = []
for dataset in data_set_replacer.keys():
    for bias_type in bias_types:
        for method in method_name_replacer.keys():
            json_file = result_path / dataset / bias_type /  "0.1"/ method / "classification_results.json"
            try:
                result_file = pd.read_json(str(json_file))
            except FileNotFoundError:
                continue
            dict_list.append(
                {
                    "Method": method, "Data Set": dataset, 
                    "AUROC Mean": result_file["random forest auroc"]["mean"], 
                    "AUROC Std": result_file["random forest auroc"]["sd"], 
                    "AUPRC Mean": result_file["random forest auprc"]["mean"], 
                    "AUPRC Std": result_file["random forest auprc"]["sd"], 
                    "Bias Type": bias_type, "Bias Strength": "0.1",
                    "Dropped Samples Mean": result_file["dropped_samples"]["mean"],
                    "Dropped Samples Std": result_file["dropped_samples"]["std"],
                }
                            )
result_df = pd.DataFrame(data=dict_list)

In [24]:
result_df = result_df.replace(method_name_replacer)
result_df

,Method,Data Set,AUROC Mean,AUROC Std,AUPRC Mean,AUPRC Std,Bias Type,Bias Strength,Dropped Samples Mean,Dropped Samples Std
0,Uniform,diabetes,0.791498,0.022072,0.371281,0.037717,less_positive_class,0.1,0.000000,0.000000
1,KMM,diabetes,0.781169,0.021943,0.357066,0.041765,less_positive_class,0.1,0.000000,0.000000
2,PSA,diabetes,0.787277,0.023148,0.367046,0.037046,less_positive_class,0.1,0.000000,0.000000
3,MRS,diabetes,0.788607,0.023040,0.367308,0.039618,less_positive_class,0.1,49.900000,32.116818
4,FW-MRS,diabetes,0.784586,0.022317,0.359473,0.040166,less_positive_class,0.1,38.636364,29.778799
5,FW-MRS$_{SVM}$,diabetes,0.786527,0.025484,0.358726,0.043561,less_positive_class,0.1,71.052632,31.438283
6,Soft-MRS,diabetes,0.787917,0.021554,0.367637,0.037645,less_positive_class,0.1,0.000000,0.000000
7,Uniform,folktables_employment,0.870601,0.010491,0.826992,0.017307,less_positive_class,0.1,0.000000,0.000000
8,KMM,folktables_employment,0.856631,0.013580,0.808776,0.021985,less_positive_class,0.1,0.000000,0.000000
9,PSA,folktables_employment,0.867401,0.010937,0.823689,0.017363,less_positive_class,0.1,0.040000,0.280000


In [25]:
for bias_type in bias_types:
    print(f"{bias_type}, {0.1}")
    for dataset in data_set_replacer.keys():
        mean_auroc_values = []
        std_auroc_values = []
        for method in result_df["Method"].unique():
            try:
                mean_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==0.1) & 
                                                    (result_df["Data Set"]==dataset)]["AUROC Mean"].iloc[0]
                mean_auroc_values.append(np.round(mean_auroc, 3))

                std_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==0.1) & 
                                                    (result_df["Data Set"]==dataset)]["AUROC Std"].iloc[0]
                std_auroc_values.append(np.round(std_auroc, 2))
            except IndexError:
                mean_auroc_values.append(0)
                std_auroc_values.append(0)

        print(f"{data_set_replacer[dataset]} \
& ${mean_auroc_values[0]}\\pm{std_auroc_values[0]}$ \
& ${mean_auroc_values[1]}\\pm{std_auroc_values[1]}$ \
& ${mean_auroc_values[2]}\\pm{std_auroc_values[2]}$ \
& ${mean_auroc_values[3]}\\pm{std_auroc_values[3]}$ \
& ${mean_auroc_values[4]}\\pm{std_auroc_values[4]}$ \
& ${mean_auroc_values[5]}\\pm{std_auroc_values[5]}$ \
& ${mean_auroc_values[6]}\\pm{std_auroc_values[6]}$ \
\\\\")
    print("\n")

less_positive_class, 0.1
Diabetes & $0.791\pm0.02$ & $0.781\pm0.02$ & $0.787\pm0.02$ & $0.789\pm0.02$ & $0.785\pm0.02$ & $0.787\pm0.03$ & $0.788\pm0.02$ \\
Employment & $0.871\pm0.01$ & $0.857\pm0.01$ & $0.867\pm0.01$ & $0.87\pm0.01$ & $0.87\pm0.01$ & $0.864\pm0.01$ & $0.862\pm0.01$ \\
Income & $0.838\pm0.01$ & $0.82\pm0.01$ & $0.831\pm0.01$ & $0.837\pm0.01$ & $0.837\pm0.01$ & $0.835\pm0.01$ & $0.827\pm0.01$ \\
Bank Marketing & $0.847\pm0.02$ & $0.832\pm0.03$ & $0.839\pm0.03$ & $0.845\pm0.02$ & $0.845\pm0.02$ & $0.84\pm0.02$ & $0.834\pm0.03$ \\
HR Analytic & $0.753\pm0.02$ & $0.749\pm0.02$ & $0.75\pm0.02$ & $0.751\pm0.02$ & $0.751\pm0.03$ & $0.75\pm0.03$ & $0.751\pm0.02$ \\
German Credit & $0.667\pm0.05$ & $0.649\pm0.05$ & $0.659\pm0.06$ & $0.672\pm0.05$ & $0.642\pm0.06$ & $0.637\pm0.06$ & $0.657\pm0.05$ \\
Breast Cancer & $0.988\pm0.01$ & $0.989\pm0.01$ & $0.988\pm0.01$ & $0.989\pm0.01$ & $0.983\pm0.01$ & $0.979\pm0.02$ & $0.989\pm0.01$ \\
Loan & $0.658\pm0.08$ & $0.61\pm0.1$ & $0.628

In [32]:
for bias_type in bias_types:
    print(f"{bias_type}, {0.1}")
    for dataset in data_set_replacer.keys():
        mean_auprc_values = []
        std_auprc_values = []
        for method in result_df["Method"].unique():
            try:
                mean_auprc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==0.1) & 
                                                    (result_df["Data Set"]==dataset)]["AUPRC Mean"].iloc[0]
                mean_auprc_values.append(np.round(mean_auprc, 3))

                std_auprc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==0.1) & 
                                                    (result_df["Data Set"]==dataset)]["AUPRC Std"].iloc[0]
                std_auprc_values.append(np.round(std_auprc, 2))
            except IndexError:
                mean_auprc_values.append(0)
                std_auprc_values.append(0)

        print(f"{data_set_replacer[dataset]} \
& ${mean_auprc_values[0]}\\pm{std_auprc_values[0]}$ \
& ${mean_auprc_values[1]}\\pm{std_auprc_values[1]}$ \
& ${mean_auprc_values[2]}\\pm{std_auprc_values[2]}$ \
& ${mean_auprc_values[3]}\\pm{std_auprc_values[3]}$ \
& ${mean_auprc_values[4]}\\pm{std_auprc_values[4]}$ \
& ${mean_auprc_values[5]}\\pm{std_auprc_values[5]}$ \
& ${mean_auprc_values[6]}\\pm{std_auprc_values[6]}$ \
& \\\\")
print("\n")

less_positive_class, 0.1
Diabetes & $0.371\pm0.04$ & $0.357\pm0.04$ & $0.367\pm0.04$ & $0.367\pm0.04$ & $0.359\pm0.04$ & $0.359\pm0.04$ & $0.368\pm0.04$ & \\
Employment & $0.827\pm0.02$ & $0.809\pm0.02$ & $0.824\pm0.02$ & $0.826\pm0.02$ & $0.824\pm0.02$ & $0.817\pm0.02$ & $0.818\pm0.02$ & \\
Income & $0.789\pm0.02$ & $0.765\pm0.02$ & $0.781\pm0.02$ & $0.788\pm0.02$ & $0.789\pm0.02$ & $0.788\pm0.02$ & $0.774\pm0.02$ & \\
Bank Marketing & $0.466\pm0.05$ & $0.446\pm0.06$ & $0.455\pm0.05$ & $0.468\pm0.05$ & $0.456\pm0.06$ & $0.44\pm0.05$ & $0.447\pm0.05$ & \\
HR Analytic & $0.455\pm0.03$ & $0.449\pm0.03$ & $0.454\pm0.03$ & $0.456\pm0.03$ & $0.451\pm0.04$ & $0.448\pm0.04$ & $0.454\pm0.04$ & \\
German Credit & $0.461\pm0.06$ & $0.443\pm0.06$ & $0.452\pm0.06$ & $0.464\pm0.07$ & $0.441\pm0.07$ & $0.441\pm0.08$ & $0.451\pm0.06$ & \\
Breast Cancer & $0.994\pm0.0$ & $0.995\pm0.0$ & $0.994\pm0.0$ & $0.995\pm0.0$ & $0.991\pm0.01$ & $0.988\pm0.01$ & $0.994\pm0.0$ & \\
Loan & $0.795\pm0.05$ & $0.776\

In [27]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for dataset in data_set_replacer.keys():
            mean_pad_values = []
            std_pad_values = []
            for method in result_df["Method"].unique():
                try:
                    mean_pad = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["SVM PAD Mean"].iloc[0]
                    mean_pad_values.append(np.round(mean_pad, 3))

                    std_pad = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["SVM PAD Std"].iloc[0]
                    std_pad_values.append(np.round(std_pad, 2))
                except IndexError:
                    mean_pad_values.append(0)
                    std_pad_values.append(0)

            print(f"{data_set_replacer[dataset]} \
& ${mean_pad_values[0]}\\pm{std_pad_values[0]}$ \
& ${mean_pad_values[1]}\\pm{std_pad_values[1]}$ \
& ${mean_pad_values[2]}\\pm{std_pad_values[2]}$ \
& ${mean_pad_values[3]}\\pm{std_pad_values[3]}$ \
& ${mean_pad_values[4]}\\pm{std_pad_values[4]}$ \
& ${mean_pad_values[5]}\\pm{std_pad_values[5]}$ \
& \\\\")
        print("\n")

less_positive_class, 0.1


KeyError: 'SVM PAD Mean'

In [ ]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for dataset in data_set_replacer.keys():
            mean_domain_values = []
            std_domain_values = []
            for method in result_df["Method"].unique():
                try:
                    mean_domain = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["RF Domain AUROC Mean"].iloc[0]
                    mean_domain_values.append(np.round(mean_domain, 3))

                    std_domain = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["RF Domain AUROC Std"].iloc[0]
                    std_domain_values.append(np.round(std_domain, 2))
                except IndexError:
                    mean_domain_values.append(0)
                    std_domain_values.append(0)

            print(f"{data_set_replacer[dataset]} \
& ${mean_domain_values[0]}\\pm{std_domain_values[0]}$ \
& ${mean_domain_values[1]}\\pm{std_domain_values[1]}$ \
& ${mean_domain_values[2]}\\pm{std_domain_values[2]}$ \
& ${mean_domain_values[3]}\\pm{std_domain_values[3]}$ \
& ${mean_domain_values[4]}\\pm{std_domain_values[4]}$ \
& ${mean_domain_values[5]}\\pm{std_domain_values[5]}$ \
& \\\\")
        print("\n")

less_positive_class, 0.1
Employment & $0.63\pm0.01$ & $0.525\pm0.01$ & $0.528\pm0.01$ & $0.581\pm0.03$ & $0.54\pm0.03$ & $0.553\pm0.01$ & \\
Income & $0.613\pm0.01$ & $0.528\pm0.01$ & $0.519\pm0.01$ & $0.564\pm0.03$ & $0.557\pm0.03$ & $0.568\pm0.02$ & \\
Breast Cancer & $0.731\pm0.02$ & $0.592\pm0.03$ & $0.583\pm0.03$ & $0.65\pm0.03$ & $0.666\pm0.03$ & $0.675\pm0.05$ & \\
HR Analytic & $0.533\pm0.01$ & $0.514\pm0.01$ & $0.511\pm0.0$ & $0.526\pm0.01$ & $0.526\pm0.01$ & $0.524\pm0.01$ & \\
Loan & $0.57\pm0.03$ & $0.585\pm0.02$ & $0.554\pm0.02$ & $0.554\pm0.02$ & $0.553\pm0.01$ & $0.549\pm0.02$ & \\
Diabetes & $0.525\pm0.01$ & $0.522\pm0.01$ & $0.512\pm0.0$ & $0.52\pm0.01$ & $0.52\pm0.01$ & $0.517\pm0.01$ & \\
German Credit & $0.538\pm0.02$ & $0.547\pm0.02$ & $0.533\pm0.01$ & $0.533\pm0.01$ & $0.529\pm0.01$ & $0.531\pm0.01$ & \\
Bank Marketing & $0.523\pm0.01$ & $0.521\pm0.01$ & $0.512\pm0.0$ & $0.517\pm0.01$ & $0.517\pm0.01$ & $0.517\pm0.01$ & \\




In [ ]:
result_df.round(3)

,Method,Data Set,AUROC Mean,AUROC Std,AUPRC Mean,AUPRC Std,Bias Type,Bias Strength,Dropped Samples Mean,Dropped Samples Std,SVM PAD Mean,SVM PAD Std,RF Domain AUROC Mean,RF Domain AUROC Std
0,Uniform,folktables_employment,0.871,0.010,0.827,0.017,less_positive_class,0.1,0.000,0.000,0.357,0.037,0.630,0.010
1,KMM,folktables_employment,0.857,0.014,0.809,0.022,less_positive_class,0.1,0.000,0.000,0.099,0.023,0.525,0.009
2,PSA,folktables_employment,0.867,0.011,0.824,0.017,less_positive_class,0.1,0.040,0.280,0.042,0.012,0.528,0.010
3,MRS,folktables_employment,0.870,0.010,0.825,0.017,less_positive_class,0.1,270.300,99.922,0.226,0.090,0.581,0.031
4,FW-MRS,folktables_employment,0.864,0.012,0.819,0.018,less_positive_class,0.1,381.786,82.735,0.114,0.071,0.540,0.026
5,FW-MRS$_{SVM}$,folktables_employment,0.835,0.016,0.776,0.027,less_positive_class,0.1,271.200,29.267,0.100,0.037,0.553,0.014
6,Uniform,folktables_income,0.838,0.013,0.789,0.020,less_positive_class,0.1,0.000,0.000,0.353,0.041,0.613,0.014
7,KMM,folktables_income,0.820,0.014,0.765,0.020,less_positive_class,0.1,0.000,0.000,0.093,0.026,0.528,0.007
8,PSA,folktables_income,0.831,0.014,0.781,0.022,less_positive_class,0.1,0.160,0.703,0.045,0.014,0.519,0.008
9,MRS,folktables_income,0.837,0.013,0.787,0.019,less_positive_class,0.1,266.000,83.970,0.218,0.089,0.564,0.029


In [ ]:
result_df["Rank AUROC"] = result_df.round(3).groupby("Data Set")["AUROC Mean"].rank(ascending=False)
result_df["Rank AUPRC"] = result_df.round(3).groupby("Data Set")["AUPRC Mean"].rank(ascending=False)
result_df["Rank PAD"] = result_df.round(3).groupby("Data Set")["SVM PAD Mean"].rank(ascending=True)
result_df["Rank Domain"] = result_df.round(3).groupby("Data Set")["RF Domain AUROC Mean"].rank(ascending=True)
result_df[["Method", "Rank AUROC", "Rank AUPRC", "Rank PAD", "Rank Domain"]].groupby("Method").mean()

,Rank AUROC,Rank AUPRC,Rank PAD,Rank Domain
Method,,,,
FW-MRS,3.6250,4.0000,3.8125,3.000
FW-MRS$_{SVM}$,5.1875,5.2500,2.5000,3.125
KMM,4.8125,4.6875,2.5625,3.625
MRS,2.1875,1.8750,4.8750,3.750
PSA,3.6250,3.4375,1.2500,1.750
Uniform,1.5625,1.7500,6.0000,5.750
